# 🚀 Mastering AI Agents: Multi-Agent Orchestration with Google ADK! 🚀

Welcome, Agent Architect to LAB2! This notebook is your guide to Build a Foundational Agent. Create a simple but effective AI agent from scratch using the Google Agent Development Kit (ADK).


giving your AI agents two essential superpowers: custom tools and conversational memory.

The series for Labs and Goals:
- **[HERE]>>>Lab2: Build a Foundational Agent**: Create a simple but effective AI agent from scratch using the Google Agent Development Kit (ADK).

- **Lab 3: Grant New Skills with Custom Tools**: Teach an agent to perform new tasks by connecting it to external APIs, like a real-time weather service.

- **Lab 4: Create a Team of Agents**: Assemble a multi-agent system where a primary agent can delegate specialized tasks to other agents.

- **Demo: Master Conversational Memory**: Understand the critical role of Sessions in enabling agents to remember previous interactions, handle feedback, and carry on a coherent conversation.


Let's get this adventure started!

Credit: Notebook content adapted from Qingyue (Annie) Wang, a developer advocate and AI engineer at **Google**, who passionate about helping developers build with AI and cloud technologies.



-------------
### 🎁 🛑 Important Prerequisite: Setup Your Environment! 🛑 🎁
-----------------------------------------------------------------------------

👉 **Get Your API Key HERE**: [Google AI Studio](https://aistudio.google.com/app/apikey)

 -----------------------------------------------------------------------------


## Lab 1: Setup & Authentication 🔑

First things first, let's get all our tools ready. This step installs the necessary libraries and securely configures your Google API key so your agents can access the power of Gemini.

In [ ]:
# --- Lab Cell ID 1----#
!pip install google-adk google-generativeai -q

# --- Import all necessary libraries ---
import os
import sys
import json
import asyncio
import random
import string
from uuid import uuid4
from typing import Any, List

import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, Markdown, display

# --- ADK, Agent, and Evaluation Components ---
from google.adk.agents import Agent
from google.adk.events import Event
from google.adk.runners import Runner
import google.adk as adk
from google.adk.tools import google_search
from google.adk.sessions import InMemorySessionService, Session
from google.genai import types
from google.genai.types import Content, Part


print("✅ All libraries are ready to go!")



✅ All libraries are ready to go!


/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


### Configure Your API Key
To use Gemini models, you need an API key from [Google AI Studio](https://aistudio.google.com/app/apikey). This section securely collects your key and configures it for the ADK.


In [ ]:
# --- Lab Cell ID 2----#
# --- API Key Configuration ---
from google.colab import userdata

# Use Colab Secrets (recommended option)
# Go to the 🔑 icon in the left sidebar, add a secret named GOOGLE_API_KEY
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    print("✅ API key loaded from Colab Secrets.")
except Exception:
    # Option 2: Paste it directly (less secure but fine for learning)
    import getpass
    GOOGLE_API_KEY = getpass.getpass("🔑 Enter your Google AI Studio API key: ")
    print("✅ API key entered manually.")


✅ API key loaded from Colab Secrets.


In [ ]:
# --- Lab Cell ID 3----#
# --- Set Environment Variables for ADK ---
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"

print(f"✅ API key configured (starts with '{GOOGLE_API_KEY[:6]}...')")
print("✅ Using Google AI Studio (not Vertex AI).")


✅ API key configured (starts with 'AQ.Ab8...')
✅ Using Google AI Studio (not Vertex AI).


---
## Lab2: Your First Agent - The Day Trip Genie 🧞

Meet your first creation! The `day_trip_agent` is a simple but powerful assistant. We're making it a little smarter by teaching it to understand **budget constraints**.

* **Agent**: The brain of the operation, defined by its instructions, tools, and the AI model it uses.
* **Session**: The conversation history. For this simple agent, it's just a container for a single request-response.
* **Runner**: The engine that connects the `Agent` and the `Session` to process your request and get a response.

```

    +------------------+
    |   User Input     |
    |------------------|
    |  Mood            |
    |  Interests       |
    |  Budget          |
    +------------------+

            |
            ▼
+--------------------------------------------------+
|         Spontaneous Day Trip Agent 🤖            |
|--------------------------------------------------|
|  Model: gemini-2.5-flash                         |
|  Description:                                    |
|   Generates full-day trip itineraries based on   |
|   mood, interests, and budget                    |
|--------------------------------------------------|
|  🔧 Tools:                                       |
|   - Google Search                                |
|--------------------------------------------------|
|  🧠 Capabilities:                                |
|   - Budget Awareness (cheap / splurge)           |
|   - Mood Matching (adventurous, relaxing, etc.)  |
|   - Real-Time Info (hours, events)               |
|   - Morning / Afternoon / Evening plan           |
+--------------------------------------------------+
            |
            ▼
+--------------------------------------------------+
|             Output: Markdown Itinerary           |
|--------------------------------------------------|
| - Time blocks (Morning / Afternoon / Evening)    |
| - Venue names with links and hours               |
| - Budget-matching activities                     |
+--------------------------------------------------+
```


In [ ]:
# --- Lab Cell ID 4----#

# --- Agent Definition ---

def create_day_trip_agent():
    """Create the Spontaneous Day Trip Generator agent"""
    return Agent(
        name="day_trip_agent",
        model="gemini-2.5-flash",
        description="Agent specialized in generating spontaneous full-day itineraries based on mood, interests, and budget.",
        instruction="""
        You are the "Spontaneous Day Trip" Generator 🚗 - a specialized AI assistant that creates engaging full-day itineraries.

        Your Mission:
        Transform a simple mood or interest into a complete day-trip adventure with real-time details, while respecting a budget.

        Guidelines:
        1. **Budget-Aware**: Pay close attention to budget hints like 'cheap', 'affordable', or 'splurge'. Use Google Search to find activities (free museums, parks, paid attractions) that match the user's budget.
        2. **Full-Day Structure**: Create morning, afternoon, and evening activities.
        3. **Real-Time Focus**: Search for current operating hours and special events.
        4. **Mood Matching**: Align suggestions with the requested mood (adventurous, relaxing, artsy, etc.).

        RETURN itinerary in MARKDOWN FORMAT with clear time blocks and specific venue names.
        """,
        tools=[google_search]
    )

day_trip_agent = create_day_trip_agent()
print(f"🧞 Agent '{day_trip_agent.name}' is created and ready for adventure!")

🧞 Agent 'day_trip_agent' is created and ready for adventure!


In [ ]:
# --- Lab Cell ID 5 ----# Share with Scenario 3

# --- A Helper Function to Run Our Agents ---
# We'll use this function throughout the notebook to make running queries easy.

async def run_agent_query(agent: Agent, query: str, session: Session, user_id: str, is_router: bool = False):
    """Initializes a runner and executes a query for a given agent and session."""
    print(f"\n🚀 Running query for agent: '{agent.name}' in session: '{session.id}'...")

    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )

    final_response = ""
    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=Content(parts=[Part(text=query)], role="user")
        ):
            if not is_router:
                # Let's see what the agent is thinking!
                print(f"EVENT: {event}")
            if event.is_final_response():
                final_response = event.content.parts[0].text
    except Exception as e:
        final_response = f"An error occurred: {e}"

    if not is_router:
     print("\n" + "-"*50)
     print("✅ Final Response:")
     display(Markdown(final_response))
     print("-"*50 + "\n")

    return final_response



In [ ]:
# --- Lab Cell ID 6 ----#

# For Part2: You also have to run This Cell together with Cell 1 and Cell 2

# --- Initialize our Session Service ---
# This one service will manage all the different sessions in our notebook.
session_service = InMemorySessionService()
my_user_id = "adk_adventurer_001"

In [ ]:
# ---  Lab Cell ID 7 ----#

# --- [MAIN] Let's test the Day Trip Genie! ---

async def run_day_trip_genie():
    # Create a new, single-use session for this query
    day_trip_session = await session_service.create_session(
        app_name=day_trip_agent.name,
        user_id=my_user_id
    )

    # Note the new budget constraint in the query!
    query = "Plan a relaxing and artsy day trip near Sunnyvale, CA. Keep it affordable!"
    print(f"🗣️ User Query: '{query}'")

    await run_agent_query(day_trip_agent, query, day_trip_session, my_user_id)

await run_day_trip_genie()

🗣️ User Query: 'Plan a relaxing and artsy day trip near Sunnyvale, CA. Keep it affordable!'

🚀 Running query for agent: 'day_trip_agent' in session: '03dfc240-00cf-49ea-b7fa-3c99b0756f82'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Sunnyvale, California, offers a delightful blend of art and tranquility, perfect for an affordable and relaxing day trip. This itinerary focuses on free or low-cost activities that align with your mood and budget.

---

## Relaxing & Artsy Day Trip near Sunnyvale, CA

**Budget:** Affordable (Free activities, packed lunch/casual dining options)

### Morning (11:00 AM - 1:30 PM): Immerse in Local Art

*   **11:00 AM - 1:00 PM: Triton Museum of Art (Santa Clara)**
    Start your day with a visit to the **Triton Museum of Art** in Santa Clara. This museum offers free admission and free parking, making it a perfect budget-friendly choice. The museum showcases contemporary and historical works with an emphasis on l

Sunnyvale, California, offers a delightful blend of art and tranquility, perfect for an affordable and relaxing day trip. This itinerary focuses on free or low-cost activities that align with your mood and budget.

---

## Relaxing & Artsy Day Trip near Sunnyvale, CA

**Budget:** Affordable (Free activities, packed lunch/casual dining options)

### Morning (11:00 AM - 1:30 PM): Immerse in Local Art

*   **11:00 AM - 1:00 PM: Triton Museum of Art (Santa Clara)**
    Start your day with a visit to the **Triton Museum of Art** in Santa Clara. This museum offers free admission and free parking, making it a perfect budget-friendly choice. The museum showcases contemporary and historical works with an emphasis on local Bay Area artists. Take your time exploring the galleries and enjoy the serene outdoor sculpture garden, which provides a relaxing atmosphere.
    *Location: 1505 Warburton Ave, Santa Clara, CA 95050*
    *Cost: Free*
    *Operating Hours: Tuesday - Sunday: 11:00 a.m. to 4:30 p.m. (Closed Mondays & holidays)*

*   **1:00 PM - 1:30 PM: Travel to Sunnyvale**
    A short drive will bring you back to Sunnyvale for the next part of your artistic and relaxing day.

### Afternoon (1:30 PM - 5:30 PM): Outdoor Art and Nature's Calm

*   **1:30 PM - 2:30 PM: Affordable Lunch Break**
    For an affordable lunch, consider packing a picnic to enjoy at one of Sunnyvale's many parks, or grab a casual, budget-friendly bite from a local deli or eatery in Downtown Sunnyvale.
    *Cost: $ (Packed lunch or casual dining)*

*   **2:30 PM - 4:00 PM: Sunnyvale Public Art Self-Guided Tour**
    After lunch, embark on a self-guided tour of Sunnyvale's vibrant public art collection. The city boasts over 200 public art pieces, including the "Sun Flair" sculpture program, which features 26 identical sun sculptures transformed by local artists and installed in various city parks. You can find walking tour maps (PDFs) on the City of Sunnyvale's website, covering areas like the Civic Center, Washington Park, and Downtown. This allows for a relaxing stroll while discovering unique artistic expressions.
    *Location: Various parks and public spaces throughout Sunnyvale (e.g., Downtown, Washington Park)*
    *Cost: Free*

*   **4:00 PM - 5:30 PM: Relax at Baylands Park**
    Transition to **Baylands Park** for an unwinding experience amidst nature. This expansive park offers plenty of breathing space, walking paths, and unique vantage points of the San Francisco Bay. It's an excellent spot for birdwatching or simply enjoying a leisurely stroll and the fresh air.
    *Location: 999 E Caribbean Dr, Sunnyvale, CA 94089*
    *Cost: Free (parking may have a fee, but often free for Sunnyvale Public Library cardholders)*

### Evening (5:30 PM onwards): Sunset and Serenity

*   **5:30 PM onwards: Sunset Views at Baylands Park or Stroll on Murphy Avenue**
    Conclude your relaxing and artsy day by returning to **Baylands Park** to catch the sunset over the San Francisco Bay. The tranquil atmosphere provides a perfect end to your day. Alternatively, if you prefer a more urban vibe, take a leisurely evening stroll along **Historic Murphy Avenue** in Downtown Sunnyvale. You can admire any remaining public art installations, enjoy the historic architecture, and soak in the evening ambiance.
    *Location: Baylands Park or Historic Murphy Avenue, Sunnyvale*
    *Cost: Free*

---

Enjoy your affordable, relaxing, and artsy day trip near Sunnyvale!

--------------------------------------------------



---
## 🎉 Congratulations! 🎉

Congratulations on completing your ADK adventure into Tools and Memory! You've taken a massive leap from building single-shot agents to creating dynamic, stateful AI systems.

Let's recap the powerful concepts you've mastered:

- **Fundamental Agent & Tools**: You started by building a "Day Trip Genie" and equipped it with its first tool, GoogleSearch.



## 📝 Assignment: Extend the Day Trip Genie

Now that you've completed Lab 2, let's put your understanding to the test by extending the `day_trip_agent`.

### Task 1: Enhance the Agent and Craft a Specific Query

Modify the `create_day_trip_agent` function and then craft a specific query to test its enhancements.

1.  **Enhance the Agent**:
    *   **Update Description**: Change the `description` to emphasize its ability to suggest *less common* or *unique* destinations.
    *   **Refine Instructions**: Add a new guideline to the `instruction` that tells the agent to prioritize activities or locations that are "off the beaten path" or "hidden gems" when available.
2.  **Craft a Query**: After modifying the agent, create a user query that asks for an "adventurous and scenic day trip near a major city of your choice (e.g., San Francisco, Los Angeles), but with a *very strict budget* (e.g., 'no more than $20 for activities and food')." You will then run this query with your modified agent.

---